# Assignment: Word2Vec + Sentence Transformers on Twitter Corpus

This notebook uses `lecture4/twitter_training.csv` as the corpus and dataset for classification.

Tasks covered:
1. Train Word2Vec embeddings with Gensim
2. Fine-tune Word2Vec with additional corpus split
3. Find most similar words for 5 words
4. Run word analogies
5. Visualize embeddings using PCA
6. Train classifiers: BoW, TF-IDF, Avg Word2Vec, Sentence-Transformer
7. Compare results with previous baseline models

In [1]:
# If needed, uncomment and run once
# %pip install -q pandas numpy matplotlib nltk gensim scikit-learn sentence-transformers

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from gensim.models import Word2Vec

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

try:
    from sentence_transformers import SentenceTransformer
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'])
    from sentence_transformers import SentenceTransformer

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

print('Libraries imported successfully.')

In [ ]:
DATA_PATH = '../lecture4/twitter_training.csv'

# Expected structure from this dataset: id, entity, sentiment, tweet_text
df = pd.read_csv(
    DATA_PATH,
    header=None,
    names=['tweet_id', 'entity', 'sentiment', 'text'],
    usecols=[0, 1, 2, 3],
    engine='python',
    on_bad_lines='skip'
)

df = df.dropna(subset=['text', 'sentiment']).copy()
df['text'] = df['text'].astype(str).str.strip()
df['sentiment'] = df['sentiment'].astype(str).str.strip()
df = df[df['text'].str.len() > 2]

# Keep a manageable sample for fast notebook execution
MAX_ROWS = 20000
if len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=42)

df = df.reset_index(drop=True)
print('Loaded rows:', len(df))
print('Sentiment distribution:')
print(df['sentiment'].value_counts())
df.head()

In [ ]:
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [tok for tok in tokens if tok.isalpha() and tok not in stop_words and len(tok) > 2]
    return tokens

df['tokens'] = df['text'].apply(preprocess_text)
df = df[df['tokens'].str.len() > 0].reset_index(drop=True)
df['clean_text'] = df['tokens'].apply(lambda x: ' '.join(x))

print('Rows after preprocessing:', len(df))
print('Example tokens:', df['tokens'].iloc[0][:15])

In [ ]:
# Train Word2Vec on 80% then fine-tune on remaining 20%
tokenized_sentences = df['tokens'].tolist()
split_idx = int(0.8 * len(tokenized_sentences))
base_sentences = tokenized_sentences[:split_idx]
additional_sentences = tokenized_sentences[split_idx:]

w2v_model = Word2Vec(
    sentences=base_sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=10
)

if len(additional_sentences) > 0:
    w2v_model.build_vocab(additional_sentences, update=True)
    w2v_model.train(additional_sentences, total_examples=len(additional_sentences), epochs=5)

w2v_model.save('word2vec_twitter.model')
print('Saved: word2vec_twitter.model')
print('Vocabulary size:', len(w2v_model.wv.index_to_key))

In [ ]:
# 5 words in corpus: most similar words
candidate_words = ['game', 'love', 'good', 'bad', 'twitter', 'borderlands', 'happy', 'sad', 'play']
query_words = [w for w in candidate_words if w in w2v_model.wv.key_to_index][:5]

if len(query_words) < 5:
    extras = [w for w in w2v_model.wv.index_to_key if w not in query_words]
    query_words.extend(extras[:(5 - len(query_words))])

print('Query words used:', query_words)
for word in query_words:
    print(f'\nMost similar words to {word}:')
    for sim_word, score in w2v_model.wv.most_similar(word, topn=5):
        print(f'  {sim_word:<15} {score:.4f}')

In [ ]:
# Word analogies: a - b + c
analogy_sets = [
    ('good', 'bad', 'great'),
    ('love', 'hate', 'like'),
    ('happy', 'sad', 'excited'),
    ('win', 'lose', 'victory'),
    ('best', 'worst', 'better')
]

for a, b, c in analogy_sets:
    print(f'\nAnalogy: {a} - {b} + {c}')
    if all(w in w2v_model.wv.key_to_index for w in [a, b, c]):
        result = w2v_model.wv.most_similar(positive=[a, c], negative=[b], topn=3)
        for word, score in result:
            print(f'  {word:<15} {score:.4f}')
    else:
        missing = [w for w in [a, b, c] if w not in w2v_model.wv.key_to_index]
        print('  Missing words in vocabulary:', missing)

In [ ]:
# PCA visualization (plot top 150 words for readability)
words = w2v_model.wv.index_to_key[:150]
vectors = w2v_model.wv[words]

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(vectors)

plt.figure(figsize=(14, 10))
plt.scatter(coords[:, 0], coords[:, 1], alpha=0.7)
for i, word in enumerate(words):
    plt.annotate(word, (coords[i, 0], coords[i, 1]), fontsize=8)
plt.title('Word2Vec Embeddings (Twitter Corpus) - PCA 2D')
plt.xlabel('PCA-1')
plt.ylabel('PCA-2')
plt.grid(alpha=0.3)
plt.show()

## Classification and Comparison

Previous-style baselines: CountVectorizer and TF-IDF.

Embedding models:
- Average Word2Vec sentence vectors + LogisticRegression
- Sentence-Transformer embeddings + LogisticRegression

In [ ]:
X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train size:', len(X_train), '| Test size:', len(X_test))
print('Classes:', sorted(y.unique().tolist()))

In [ ]:
# Baseline 1: CountVectorizer + LogisticRegression
cv = CountVectorizer(max_features=20000)
X_train_cv = cv.fit_transform(X_train)
X_test_cv = cv.transform(X_test)

clf_cv = LogisticRegression(max_iter=1000)
clf_cv.fit(X_train_cv, y_train)
pred_cv = clf_cv.predict(X_test_cv)
acc_cv = accuracy_score(y_test, pred_cv)

print('CountVectorizer Accuracy:', round(acc_cv, 4))
print(classification_report(y_test, pred_cv))

In [ ]:
# Baseline 2: TF-IDF + LogisticRegression
tfidf = TfidfVectorizer(max_features=20000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

clf_tfidf = LogisticRegression(max_iter=1000)
clf_tfidf.fit(X_train_tfidf, y_train)
pred_tfidf = clf_tfidf.predict(X_test_tfidf)
acc_tfidf = accuracy_score(y_test, pred_tfidf)

print('TF-IDF Accuracy:', round(acc_tfidf, 4))
print(classification_report(y_test, pred_tfidf))

In [ ]:
# Model 3: Average Word2Vec + LogisticRegression
def sentence_avg_vector(tokens, model, dim=100):
    vecs = [model.wv[t] for t in tokens if t in model.wv.key_to_index]
    if len(vecs) == 0:
        return np.zeros(dim)
    return np.mean(vecs, axis=0)

train_tokens = X_train.apply(preprocess_text)
test_tokens = X_test.apply(preprocess_text)

X_train_w2v = np.vstack([sentence_avg_vector(t, w2v_model, dim=100) for t in train_tokens])
X_test_w2v = np.vstack([sentence_avg_vector(t, w2v_model, dim=100) for t in test_tokens])

clf_w2v = LogisticRegression(max_iter=1000)
clf_w2v.fit(X_train_w2v, y_train)
pred_w2v = clf_w2v.predict(X_test_w2v)
acc_w2v = accuracy_score(y_test, pred_w2v)

print('Average Word2Vec Accuracy:', round(acc_w2v, 4))
print(classification_report(y_test, pred_w2v))

In [ ]:
# Model 4: Sentence-Transformer + LogisticRegression
sbert = SentenceTransformer('all-MiniLM-L6-v2')

X_train_sbert = sbert.encode(X_train.tolist(), show_progress_bar=True)
X_test_sbert = sbert.encode(X_test.tolist(), show_progress_bar=True)

clf_sbert = LogisticRegression(max_iter=1000)
clf_sbert.fit(X_train_sbert, y_train)
pred_sbert = clf_sbert.predict(X_test_sbert)
acc_sbert = accuracy_score(y_test, pred_sbert)

print('Sentence-Transformer Accuracy:', round(acc_sbert, 4))
print(classification_report(y_test, pred_sbert))

In [ ]:
results = pd.DataFrame({
    'Model': [
        'CountVectorizer + LogisticRegression',
        'TF-IDF + LogisticRegression',
        'Average Word2Vec + LogisticRegression',
        'Sentence-Transformer + LogisticRegression'
    ],
    'Accuracy': [acc_cv, acc_tfidf, acc_w2v, acc_sbert]
}).sort_values('Accuracy', ascending=False).reset_index(drop=True)

print('Model comparison (higher is better):')
results

### Notes
- This notebook now uses your course Twitter dataset as requested.
- Any analogy pair with missing words reports missing vocabulary instead of failing.
- For exact reproducibility, keep `random_state=42`.